# Digital Cash Management Forecast: lab (data scientist)

Explore the data and try models here without touching production: nothing in this notebook changes the flow, the
saved models or the production model.

1. **EDA**: the daily flows per category, totals over time, seasonality, and one client day by day.
2. **Models to try**: several forecasters trained and tested side by side on the same test as champion / challenger
   (trained without the last 6 months before the cut-off, tested on them), next to the production model and the
   simple guess (the same as the month before), with the decision champion / challenger would take for each.
   Each try is recorded in Experiment Tracking, experiment `cashflow_lab` (Lab → Experiment tracking); lab tries
   are never deployed.
3. **Put one in production.**

**Kernel:** the code env `cashflow_forecast`. If it is not in the kernel list, ask whoever manages it to enable its
Jupyter support; meanwhile, the recipe `try_forecaster` tries one forecaster without a notebook (set the project
variable `lab_forecaster` and build `lab_results`, Lab zone).

In the sandbox the data is simulated: there, the scores show that the code works, not which model is better.

In [ ]:
import calendar
import importlib
import logging
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="google")      # third-party notices on import
logging.getLogger("matplotlib").setLevel(logging.ERROR)
try:
    import mlflow
    import sklearn
except ImportError:
    raise RuntimeError("Use the kernel of the code env cashflow_forecast (Kernel > Change kernel). If it is not in the "
                       "list, ask whoever manages that code env to enable its Jupyter support; meanwhile, set the "
                       "project variable lab_forecaster and build lab_results (Lab zone).")
import dataiku

warnings.filterwarnings("ignore", category=DeprecationWarning)    # third-party notices; after dataiku, which shows them
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import cashflow
import forecasters
import models
import tracking

for module in (cashflow, models, tracking, forecasters):     # pick up changes to the project library without a restart
    importlib.reload(module)
from cashflow import FORECAST_COLUMNS, HORIZONS, TOTAL, cutoff_month, monthly_matrix, monthly_series, read_dataset
from forecasters import RecentAverage6
from models import RecentAverage, lab_compare

pd.set_option("display.max_columns", 30)
cutoff = cutoff_month()
daily = read_dataset("daily_flows")          # interface 1: one row per client, day and category
clients = read_dataset("clients")            # one row per business client
monthly = monthly_series(daily, cutoff)      # the net cash flow per client, category (and TOTAL) and month
print(f"daily_flows: {len(daily):,} rows, {daily['client_no'].nunique()} clients, {daily['category'].nunique()} "
      f"categories, {pd.to_datetime(daily['day']).min():%Y-%m-%d} to {pd.to_datetime(daily['day']).max():%Y-%m-%d}; "
      f"cut-off month {cutoff:%Y-%m}")

## 1. EDA

The data every model gets (interface 1): `daily_flows`, one row per client, day and category with money in and
money out, and `clients`. The models predict, per client, per category and in TOTAL (all categories together), the
net cash flow (money in minus money out) of each of the next 3 months. First, per category and per segment
(**negative_months**: the share of months where more money went out than came in).

In [ ]:
# Per category: how many clients use it, money in and out per month, and how often a client's month is negative
d = daily.assign(month=cashflow.month_start(daily["day"]).values)
per_month = d.groupby(["category", "client_no", "month"])[["money_in", "money_out"]].sum().reset_index()
overview = per_month.groupby("category").agg(clients=("client_no", "nunique"), money_in=("money_in", "mean"),
                                             money_out=("money_out", "mean"))
overview["negative_months"] = (monthly[monthly["category"] != TOTAL].assign(neg=lambda m: m["net"] < 0)
                               .groupby("category")["neg"].mean())
display(overview.round({"money_in": 0, "money_out": 0, "negative_months": 3}).sort_values("clients", ascending=False))

# Per segment, the client's whole net cash flow (TOTAL)
total = monthly[monthly["category"] == TOTAL].merge(clients[["client_no", "segment"]], on="client_no")
display(total.groupby("segment").agg(clients=("client_no", "nunique"), months=("month", "size"),
                                     net=("net", "mean"), negative_months=("net", lambda s: (s < 0).mean()))
        .round({"net": 0, "negative_months": 3}))

Net cash flow per category over time, and seasonality. Ramadan and Eid move about 11 days earlier each year, so a
calendar-month profile blurs them.

In [ ]:
# All clients per month: the net cash flow of each category (million SAR)
by_category = (monthly[monthly["category"] != TOTAL].pivot_table(index="month", columns="category", values="net",
                                                                  aggfunc="sum") / 1e6)
ax = by_category.plot(figsize=(12, 4.5), title="Net cash flow per category, all clients (million SAR)")
ax.axhline(0, color="grey", linewidth=0.5)
ax.set_xlabel("")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

# Seasonality without the trend: money in and out each month / the average of the 12 months before (1 = usual)
d = daily.assign(month=cashflow.month_start(daily["day"]).values)
totals = d.groupby("month")[["money_in", "money_out"]].sum()
ratio = totals / totals.rolling(12).mean().shift(1)
profile = ratio.groupby(ratio.index.month).mean()
profile.index = [calendar.month_abbr[m] for m in profile.index]
ax = profile.plot.bar(figsize=(10, 3.5), title="Seasonality: month / the 12 months before, all clients (1 = usual)")
ax.axhline(1, color="grey", linewidth=0.5)
plt.show()

One client day by day (what the models see) and its monthly totals (what they predict).

In [ ]:
# One client, day by day: what the models see (daily net, all categories), and the monthly totals they predict
client_no = monthly[monthly["category"] == TOTAL].groupby("client_no")["net"].size().idxmax()   # longest history
one = daily[daily["client_no"] == client_no].assign(net=lambda d: d["money_in"] - d["money_out"])
one = one.groupby(pd.to_datetime(one["day"]))["net"].sum().loc[lambda s: s.index > cutoff - pd.DateOffset(months=6)]
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 3.5))
(one / 1e3).plot(ax=a, style=".", title=f"Client {client_no}: daily net, last 6 months (thousand SAR)")
months = monthly[(monthly["client_no"] == client_no) & (monthly["category"] == TOTAL)].set_index("month")["net"]
(months.tail(24) / 1e3).plot.bar(ax=b, title="Monthly net, last 24 months (thousand SAR)")
b.set_xticklabels([f"{m:%y-%m}" for m in months.tail(24).index], rotation=90)
plt.tight_layout()
plt.show()

## 2. Models to try

A forecaster is a class with `name`, `params` (its settings), `fit(daily, clients, cutoff)` and
`predict(daily, clients, cutoff)`, which returns one row per client, category (and TOTAL) and months ahead with `p10`,
`ev` and `p90` in SAR (details in `models.py`, project library). `cashflow.monthly_matrix(daily, cutoff)` turns the
days into months. Compared here:

- `recent_average`: the placeholder in production (the average of the last 3 months, no machine learning)
- `recent_average_6m`: the same over 6 months (an example in `forecasters.py`)
- `lag_boosting`: an example of a small model that learns: scikit-learn gradient boosting on the last 12 months

Change them or add your own, then run the comparison. scikit-learn, pandas and matplotlib are in the code env; other
packages need the admin.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor


class LagBoosting:
    """Example of a small model that learns: scikit-learn gradient boosting on the last 12 months of each client and
    category (divided by the series' scale, so small and large clients look alike), one model each for the EV (the
    median), P10 and P90 (quantile loss). Trained on every client and category at each of the 24 cut-offs before the
    last."""
    name = "lag_boosting"
    params = {"lags": 12, "cutoffs": 24, "max_iter": 200, "learning_rate": 0.05}

    def _rows(self, wide, p, h=None):
        """Inputs at month position p (the cut-off) for months ahead h; with the answer if h is known."""
        last = wide.iloc[:, max(0, p - self.params["lags"] + 1):p + 1]
        scale = last.abs().mean(axis=1).fillna(0).clip(lower=100.0)
        X = last.div(scale, axis=0).reindex(columns=last.columns).fillna(0.0)
        X.columns = [f"lag_{len(X.columns) - i}" for i in range(len(X.columns))]
        X = X.reindex(columns=[f"lag_{k}" for k in range(self.params["lags"], 0, -1)], fill_value=0.0)
        X["total"] = (wide.index.get_level_values("category") == TOTAL).astype(int)
        X["months_known"] = last.notna().sum(axis=1).values
        X["months_ahead"] = h or 0
        return X, scale

    def fit(self, daily, clients, cutoff):
        wide = monthly_matrix(daily, cutoff)
        n, Xs, ys = wide.shape[1], [], []
        for p in range(max(0, n - 1 - self.params["cutoffs"]), n - 1):
            for h in HORIZONS:
                if p + h < n:
                    X, scale = self._rows(wide, p, h)
                    y = wide.iloc[:, p + h] / scale
                    ok = y.notna() & wide.iloc[:, p].notna()
                    Xs.append(X[ok.values])
                    ys.append(y[ok])
        X, y = pd.concat(Xs), pd.concat(ys)
        settings = {"max_iter": self.params["max_iter"], "learning_rate": self.params["learning_rate"],
                    "random_state": 0}
        self.models = {"ev": HistGradientBoostingRegressor(loss="quantile", quantile=0.5, **settings).fit(X, y),
                       "p10": HistGradientBoostingRegressor(loss="quantile", quantile=0.1, **settings).fit(X, y),
                       "p90": HistGradientBoostingRegressor(loss="quantile", quantile=0.9, **settings).fit(X, y)}
        return self

    def predict(self, daily, clients, cutoff):
        wide = monthly_matrix(daily, cutoff)
        out = []
        for h in HORIZONS:
            X, scale = self._rows(wide, wide.shape[1] - 1, h)
            rows = wide.index.to_frame(index=False).assign(months_ahead=h)
            for column, model in self.models.items():
                rows[column] = model.predict(X) * scale.values
            out.append(rows)
        return pd.concat(out, ignore_index=True)[FORECAST_COLUMNS]

In [ ]:
# Train each on the same months, test it on the last 6, next to the production model. recent_average is the
# production recipe trained again: it should match the production model's scores (a check that the test is the same).
RECORD = True       # record each try in Experiment Tracking (experiment cashflow_lab)
result = lab_compare([RecentAverage(), RecentAverage6(), LagBoosting()], record=RECORD)
print("Net cash flow per client, category and TOTAL. EV error: lower is better. P10-P90 coverage: aim 80%.")
display(result["board"])
print("EV error per months ahead (ALL: together):")
display(result["errors"])
print("P10-P90 coverage per months ahead:")
display(result["coverage"])

In [ ]:
errors = result["errors"].drop(index="ALL")
ax = errors.plot.bar(figsize=(10, 4), title="EV error per months ahead (lower is better)")
ax.set_xlabel("months ahead")
plt.tight_layout()
plt.show()

## 3. Put one in production

1. Copy its class into `forecasters.py` (project library) with the imports it needs (for example
   `import pandas as pd`, `from cashflow import FORECAST_COLUMNS, HORIZONS, TOTAL, monthly_matrix`) and add it to
   `FORECASTERS`. The setup notebook never overwrites that file.
2. Set the project variable `forecaster` to its name.
3. Run the scenario **Retrain, test and promote**. Champion / challenger promotes it only if it passes the
   promotion rules (project variables `promotion_*`) against the production model on the same 6 months; the next
   Nightly batch forecasts with it.
4. To stop training it, set `forecaster` back to `recent_average`. To use an earlier model now, activate its
   version in the saved model `cash_forecast`.